### Requisito Funcional 01 – Criação do dataset de vendas

In [59]:
import csv
import random
from datetime import datetime, timedelta

def gerar_dataset_vendas(caminho_csv="vendas.csv", n_registros=200, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos e grava em CSV."""
    random.seed(seed)

    produtos = [
        "Notebook",
        "Smartphone",
        "Tablet",
        "Monitor",
        "Teclado",
        "Mouse",
        "Headset"
    ]

    categorias = {
        "Notebook": "Computadores",
        "Smartphone": "Celulares",
        "Tablet": "Celulares",
        "Monitor": "Computadores",
        "Teclado": "Perifericos",
        "Mouse": "Perifericos",
        "Headset": "Perifericos"
    }

    precos = {
        "Notebook": 3500,
        "Smartphone": 2200,
        "Tablet": 1800,
        "Monitor": 1200,
        "Teclado": 250,
        "Mouse": 120,
        "Headset": 350
    }

    regioes = [
        "Sudeste",
        "Sul",
        "Nordeste",
        "Centro-Oeste",
        "Norte"
    ]

    data_inicio = datetime(2025, 1, 1)

    colunas = [
        "id_venda",
        "data_venda",
        "cliente",
        "produto",
        "categoria",
        "regiao",
        "quantidade",
        "preco_unitario"
    ]

    with open(caminho_csv, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=colunas)
        escritor.writeheader()

        for i in range(n_registros):
            produto = random.choice(produtos)
            categoria = categorias[produto]
            quantidade = random.randint(1, 10)
            preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)

            data = data_inicio + timedelta(days=random.randint(0, 364))
            data_txt = data.strftime("%Y-%m-%d")

            cliente = f"Cliente_{random.randint(1, 50):03d}"

            # --- sujeira proposital para a etapa de limpeza ---
            if random.random() < 0.05:
                quantidade = ""  # valor ausente

            if random.random() < 0.04:
                preco = ""  # valor ausente

            if random.random() < 0.06:
                produto = " " + produto + " "  # espacos extras

            if random.random() < 0.03:
                data_txt = "DATA INVALIDA"  # data invalida

            if random.random() < 0.10:
                cliente = random.choice([
                    cliente.upper().replace("_", "-"),
                    cliente + "!!",
                    " " + cliente,
                    cliente.replace("Cliente_", "cliente#"),
                ])

            escritor.writerow({
                "id_venda": i + 1,
                "data_venda": data_txt,
                "cliente": cliente,
                "produto": produto,
                "categoria": categoria,
                "regiao": random.choice(regioes),
                "quantidade": quantidade,
                "preco_unitario": preco,
            })

    print(f"Dataset gerado com {n_registros} registros em {caminho_csv}.")


gerar_dataset_vendas()

Dataset gerado com 200 registros em vendas.csv.


### Requisito Funcional 02 – Inspecionar e descrever os dados

In [60]:
import csv

def carregar_dataset_csv(arquivo):
    ### Carrega um arquivo CSV e retorna uma lista de dicionarios
    try:
        with open(arquivo, 'r', newline='', encoding='utf-8') as arquivo_csv:
            leitor = csv.DictReader(arquivo_csv)
            registros = list(leitor)
            return registros

    except FileNotFoundError:
        print(f"Arquivo '{arquivo}' não encontrado.")
    except Exception as e:
        print(f"Ocorreu um erro ao ler o arquivo CSV: {e}")
    finally:
        print("Processamento de leitura do csv finalizado.")

def inspecionar_dados(registros):
    ### Exibe as informacoes da lista de registros
    colunas = list(registros[0].keys()) if registros else []
    nulos = {coluna: 0 for coluna in colunas}
    for linha in registros:
        for coluna in colunas:
            if linha.get(coluna, "").strip() == "":
                nulos[coluna] += 1

    print("\n=== INSPECAO INICIAL DO DATASET ===")
    print(f"Total de registros: {len(registros)}")
    print(f"\nColunas: {colunas}")
    print(f"\nValores ausentes por coluna:\n{nulos}")
    print("\nPrimeiros registros:")
    for linha in registros[:5]:
        print(linha)


arquivo = 'vendas.csv'
registros = carregar_dataset_csv(arquivo)
inspecionar_dados(registros)

Processamento de leitura do csv finalizado.

=== INSPECAO INICIAL DO DATASET ===
Total de registros: 200

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Valores ausentes por coluna:
{'id_venda': 0, 'data_venda': 0, 'cliente': 0, 'produto': 0, 'categoria': 0, 'regiao': 0, 'quantidade': 10, 'preco_unitario': 4}

Primeiros registros:
{'id_venda': '1', 'data_venda': '2025-05-21', 'cliente': 'cliente#016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': '2', 'preco_unitario': '102.9'}
{'id_venda': '2', 'data_venda': '2025-09-16', 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': '2025-03-23', 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': '1', 'preco_unitario': '1939.76'}
{'id_venda': '4', 'data_venda'

### Requisito Funcional 03 – Limpar e tratar os dados (datetime e regex)

In [61]:
import re
from datetime import datetime

def limpar_dados(registros):
    """
    Limpa e trata a lista de registros de vendas.
    Retorna: (registros_limpos, relatorio), onde relatorio é um dicionario
    com as contagens de registros iniciais, removidos e finais.
    """
    relatorio = {"iniciais": len(registros), "removidos_data": 0,
                 "removidos_nulos": 0, "finais": 0}
    padrao_cliente = re.compile(r"^Cliente_\d{3}$", flags=re.IGNORECASE)

    limpos = []
    registros_removidos = []
    for linha in registros:

        # 1. remover espacos extras nas colunas de texto
        for chave in ("cliente", "produto", "categoria", "regiao"):
            linha[chave] = linha[chave].strip()

        # 2. converter data_venda e descartar datas invalidas
        try:
            linha["data_venda"] = datetime.strptime(linha["data_venda"], "%Y-%m-%d")
        except ValueError:
            relatorio["removidos_data"] += 1
            registros_removidos.append(linha)
            continue

        # 3. descartar nulos em quantidade e preco_unitario
        if linha["quantidade"] == "" or linha["preco_unitario"] == "":
            relatorio["removidos_nulos"] += 1
            registros_removidos.append(linha)
            continue

        # 4. ajustar os tipos numericos
        linha["quantidade"] = int(float(linha["quantidade"]))
        linha["preco_unitario"] = float(linha["preco_unitario"])

        # 5. padronizar o nome do cliente com re.sub()
        nome_limpo = re.sub(r"[^A-Za-z0-9_]", "", linha["cliente"].strip())
        linha["cliente"] = nome_limpo
        linha["cliente_fora_do_padrao"] = padrao_cliente.match(nome_limpo) is None

        limpos.append(linha)

    # 6. montar e imprimir o relatorio de limpeza
    relatorio["finais"] = len(limpos)
    print("\n=== RELATORIO DE LIMPEZA ===")
    print(relatorio)
    print("\n=== REGISTROS REMOVIDOS ===")
    for registro in registros_removidos:
        print(registro)
    return limpos, relatorio

registros_limpos, relatorio_limpeza = limpar_dados(registros)


=== RELATORIO DE LIMPEZA ===
{'iniciais': 200, 'removidos_data': 4, 'removidos_nulos': 13, 'finais': 183}

=== REGISTROS REMOVIDOS ===
{'id_venda': '2', 'data_venda': datetime.datetime(2025, 9, 16, 0, 0), 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '11', 'data_venda': 'DATA INVALIDA', 'cliente': 'Cliente_030', 'produto': 'Teclado', 'categoria': 'Perifericos', 'regiao': 'Norte', 'quantidade': '2', 'preco_unitario': '241.36'}
{'id_venda': '18', 'data_venda': datetime.datetime(2025, 1, 31, 0, 0), 'cliente': 'Cliente_015', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Centro-Oeste', 'quantidade': '2', 'preco_unitario': ''}
{'id_venda': '23', 'data_venda': datetime.datetime(2025, 3, 26, 0, 0), 'cliente': 'Cliente_025', 'produto': 'Headset', 'categoria': 'Perifericos', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '392.26'}
{'id_venda': '32', 'data_venda

### Requisito Funcional 04 – Criar colunas derivadas com transformações condicionais

In [62]:
def criar_colunas_derivadas(registros):
    meses = {
        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    }

    for registro in registros:
        registro["receita_total"] = round(registro["quantidade"] * registro["preco_unitario"], 2)

        mes = registro["data_venda"].month
        registro["mes"] = mes

        registro["mes_nome"] = meses[mes]

        if mes <= 3:
            registro["trimestre"] = "Q1"
        elif mes <= 6:
            registro["trimestre"] = "Q2"
        elif mes <= 9:
            registro["trimestre"] = "Q3"
        else:
            registro["trimestre"] = "Q4"

        registro["ano"] = registro["data_venda"].year

        receita = registro["receita_total"]

        if receita < 500:
            registro["faixa_receita_item"] = "Baixo valor"
        elif receita < 5000:
            registro["faixa_receita_item"] = "Medio valor"
        else:
            registro["faixa_receita_item"] = "Alto valor"
    return registros

registros = criar_colunas_derivadas(registros_limpos)

### Requisito Funcional 05 – Calcular métricas agregadas por Mês, Produto, Categoria e Região

In [63]:
from collections import defaultdict
import json

def calcular_metricas(registros):
    vendas_mes = defaultdict(lambda: {
        "mes_nome": "",
        "receita_total": 0,
        "quantidade": 0,
        "n_vendas": 0
    })

    vendas_produto = defaultdict(float)

    vendas_categoria = defaultdict(float)

    vendas_regiao = defaultdict(lambda: {
        "receita_total": 0,
        "n_vendas": 0
    })

    for registro in registros:
        receita = registro["receita_total"]
        quantidade = registro["quantidade"]
        mes = registro["mes"]
        nome_mes = registro["mes_nome"]
        produto = registro["produto"]
        categoria = registro["categoria"]
        regiao = registro["regiao"]

        vendas_mes[mes]["mes_nome"] = nome_mes
        vendas_mes[mes]["receita_total"] += receita
        vendas_mes[mes]["quantidade"] += quantidade
        vendas_mes[mes]["n_vendas"] += 1

        vendas_produto[produto] += receita

        vendas_categoria[categoria] += receita

        vendas_regiao[regiao]["receita_total"] += receita
        vendas_regiao[regiao]["n_vendas"] += 1

    por_mes = []

    for mes, dados in sorted(vendas_mes.items()):
        por_mes.append({
            "mes": dados["mes_nome"],
            "receita_total": dados["receita_total"],
            "quantidade": dados["quantidade"],
            "n_vendas": dados["n_vendas"]
        })

    produtos_ordenados = sorted(
        vendas_produto.items(),
        key=lambda item: item[1],
        reverse=True
    )

    top_produtos = []

    for produto, receita in produtos_ordenados[:5]:
        top_produtos.append({
            "produto": produto,
            "receita_total": receita
        })

    por_categoria = []

    for categoria, receita in sorted(
        vendas_categoria.items(),
        key=lambda item: item[1],
        reverse=True
    ):
        por_categoria.append({
            "categoria": categoria,
            "receita_total": receita
        })

    por_regiao = []

    for regiao, dados in sorted(vendas_regiao.items()):
        ticket_medio = round(dados["receita_total"] / dados["n_vendas"],2)

        por_regiao.append({
            "regiao": regiao,
            "receita_total": dados["receita_total"],
            "ticket_medio": ticket_medio
        })

    metricas = {
        "por_mes": por_mes,
        "top_produtos": top_produtos,
        "por_categoria": por_categoria,
        "por_regiao": por_regiao
    }

    return metricas

def exibir_metricas(metricas):

    print("\n=== POR MES ===")
    print(f"{'mes':<12}{'receita_total':>15}{'quantidade':>12}{'n_vendas':>10}")

    for linha in metricas["por_mes"]:
        print(
            f"{linha['mes']:<12}"
            f"{linha['receita_total']:>15.2f}"
            f"{linha['quantidade']:>12}"
            f"{linha['n_vendas']:>10}"
        )

    print("\n=== TOP PRODUTOS ===")
    print(f"{'produto':<25}{'receita_total':>15}")

    for linha in metricas["top_produtos"]:
        print(
            f"{linha['produto']:<25}"
            f"{linha['receita_total']:>15.2f}"
        )

    print("\n=== POR CATEGORIA ===")
    print(f"{'categoria':<20}{'receita_total':>15}")

    for linha in metricas["por_categoria"]:
        print(
            f"{linha['categoria']:<20}"
            f"{linha['receita_total']:>15.2f}"
        )

    print("\n=== POR REGIAO ===")
    print(f"{'regiao':<20}{'receita_total':>15}{'ticket_medio':>15}")

    for linha in metricas["por_regiao"]:
        print(
            f"{linha['regiao']:<20}"
            f"{linha['receita_total']:>15.2f}"
            f"{linha['ticket_medio']:>15.2f}"
        )

metricas = calcular_metricas(registros)
#print(json.dumps(metricas, indent=2))
exibir_metricas(metricas)


=== POR MES ===
mes           receita_total  quantidade  n_vendas
Janeiro           120866.25          92        15
Fevereiro          73895.74          70        12
Março             123869.23          98        18
Abril              80636.64          48         7
Maio              132080.62         104        19
Junho             106534.48          95        15
Julho             106667.30          84        14
Agosto             85790.38          72        12
Setembro           84081.59          89        17
Outubro           110865.86          72        17
Novembro          160297.77         136        23
Dezembro          104760.44          71        14

=== TOP PRODUTOS ===
produto                    receita_total
Notebook                       374174.85
Tablet                         335335.83
Smartphone                     303255.94
Monitor                        169554.67
Headset                         48368.37

=== POR CATEGORIA ===
categoria             receita_total
Celula

### Requisito Funcional 06 – Segmentar clientes por nível de gasto

In [64]:
def segmentar_clientes(registros):
    """
    Agrupa por cliente, soma a receita e classifica em
    Bronze / Prata / Ouro usando uma funcao lambda.
    Retorna uma lista de dicionarios: cliente, total_gasto, segmento.
    """
    classificar = lambda total: (
        "Ouro" if total > 15000 else "Prata" if total >= 5000 else "Bronze"
    )
    total_por_cliente = {}
    for linha in registros:
        total_por_cliente[linha["cliente"]] = (
        total_por_cliente.get(linha["cliente"], 0) + linha["receita_total"]
    )
    clientes = [
        {"cliente": nome, "total_gasto": total, "segmento": classificar(total)}
        for nome, total in total_por_cliente.items()
    ]

    return clientes

def exibir_segmentacao(clientes):
    clientes_ordenados = sorted(
        clientes,
        key=lambda cliente: cliente["total_gasto"],
        reverse=True
    )

    top_10 = clientes_ordenados[:10]

    distribuicao = {}

    for cliente in clientes:
        segmento = cliente["segmento"]
        distribuicao[segmento] = distribuicao.get(segmento, 0) + 1

    print("\n=== TOP 10 CLIENTES ===")
    print(f"{'cliente':<25}{'total_gasto':>15}{'segmento':>12}")

    for cliente in top_10:
        print(
            f"{cliente['cliente']:<25}"
            f"{cliente['total_gasto']:>15.2f}"
            f"{cliente['segmento']:>12}"
        )

    print("\n=== DISTRIBUIÇÃO POR SEGMENTO ===")
    print(f"{'segmento':<15}{'quantidade':>12}")

    for segmento, quantidade in distribuicao.items():
        print(f"{segmento:<15}{quantidade:>12}")

clientes = segmentar_clientes(registros)
exibir_segmentacao(clientes)


=== TOP 10 CLIENTES ===
cliente                      total_gasto    segmento
Cliente_006                     65905.09        Ouro
Cliente_036                     63433.04        Ouro
Cliente_039                     59406.90        Ouro
Cliente_020                     58459.54        Ouro
Cliente_034                     52035.60        Ouro
Cliente_009                     50827.96        Ouro
Cliente_024                     49146.94        Ouro
Cliente_019                     40731.05        Ouro
Cliente_047                     38291.74        Ouro
Cliente_012                     38111.80        Ouro

=== DISTRIBUIÇÃO POR SEGMENTO ===
segmento         quantidade
Ouro                     35
Bronze                    9
Prata                    14


### Requisito Funcional 07 – Organizar o código em funções reutilizáveis

In [65]:
# Função de ordem superior
def processar_coluna(registros, coluna, funcao_transformacao, nome_saida=None):
    """
    Aplica uma funcao de transformacao a um campo de cada registro.
    Demonstra o uso de funcoes como argumento (funcao de ordem superior).
    """
    nome_saida = nome_saida or f"{coluna}_transformado"
    for linha in registros:
        linha[nome_saida] = funcao_transformacao(linha[coluna])
    return registros


# uso com funcoes lambda em contextos distintos
registros = processar_coluna(
    registros,
    "receita_total",
    lambda x: round(x / 1000, 2),
    nome_saida="receita_em_milhares",
)

registros = processar_coluna(
    registros,
    "quantidade",
    lambda q: "Alto Volume" if q > 5 else "Baixo Volume",
    nome_saida="perfil_volume",
)

print("\nPrimeiros registros:")
for linha in registros[:5]:
    print(linha)


Primeiros registros:
{'id_venda': '1', 'data_venda': datetime.datetime(2025, 5, 21, 0, 0), 'cliente': 'cliente016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': 2, 'preco_unitario': 102.9, 'cliente_fora_do_padrao': True, 'receita_total': 205.8, 'mes': 5, 'mes_nome': 'Maio', 'trimestre': 'Q2', 'ano': 2025, 'faixa_receita_item': 'Baixo valor', 'receita_em_milhares': 0.21, 'perfil_volume': 'Baixo Volume'}
{'id_venda': '3', 'data_venda': datetime.datetime(2025, 3, 23, 0, 0), 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 'Centro-Oeste', 'quantidade': 1, 'preco_unitario': 1939.76, 'cliente_fora_do_padrao': False, 'receita_total': 1939.76, 'mes': 3, 'mes_nome': 'Março', 'trimestre': 'Q1', 'ano': 2025, 'faixa_receita_item': 'Medio valor', 'receita_em_milhares': 1.94, 'perfil_volume': 'Baixo Volume'}
{'id_venda': '4', 'data_venda': datetime.datetime(2025, 11, 6, 0, 0), 'cliente': 'Cliente_017', 'produto': 'Notebook', 'c

### Requisito Funcional 08 – Exportar resultados em CSV e JSON

In [66]:
import csv
import json
import os

def exportar_resultados(metricas, clientes):
    os.makedirs("exportacao-resultados", exist_ok=True)

    with open("exportacao-resultados/metricas_por_mes.csv", "w", newline="",
              encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(
            f,
            fieldnames=metricas["por_mes"][0].keys()
        )
        escritor.writeheader()
        escritor.writerows(metricas["por_mes"])

    with open("exportacao-resultados/segmentacao_clientes.csv", "w", newline="",
              encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(
            f,
            fieldnames=clientes[0].keys()
        )
        escritor.writeheader()
        escritor.writerows(clientes)

    caminho = "exportacao-resultados/segmentacao_clientes.json"

    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(clientes, f, indent=4, ensure_ascii=False)

    with open(caminho, "r", encoding="utf-8") as f:
        conferencia = json.load(f)

    print(f"JSON gravado e lido: {conferencia}")

exportar_resultados(metricas, clientes)


JSON gravado e lido: [{'cliente': 'cliente016', 'total_gasto': 17238.2, 'segmento': 'Ouro'}, {'cliente': 'Cliente_045', 'total_gasto': 3605.96, 'segmento': 'Bronze'}, {'cliente': 'Cliente_017', 'total_gasto': 34462.64000000001, 'segmento': 'Ouro'}, {'cliente': 'Cliente_037', 'total_gasto': 29281.2, 'segmento': 'Ouro'}, {'cliente': 'Cliente_041', 'total_gasto': 19516.449999999997, 'segmento': 'Ouro'}, {'cliente': 'Cliente_011', 'total_gasto': 32896.41, 'segmento': 'Ouro'}, {'cliente': 'Cliente_021', 'total_gasto': 18608.7, 'segmento': 'Ouro'}, {'cliente': 'Cliente_030', 'total_gasto': 13449.470000000001, 'segmento': 'Prata'}, {'cliente': 'Cliente_033', 'total_gasto': 10847.970000000001, 'segmento': 'Prata'}, {'cliente': 'Cliente_008', 'total_gasto': 12025.04, 'segmento': 'Prata'}, {'cliente': 'Cliente_006', 'total_gasto': 65905.09, 'segmento': 'Ouro'}, {'cliente': 'Cliente_028', 'total_gasto': 9653.96, 'segmento': 'Prata'}, {'cliente': 'Cliente_029', 'total_gasto': 36318.049999999996, '

### Requisito Funcional 09 – Executar o fluxo completo (Ponto de Entrada)

In [69]:
def main():
    """Executa o fluxo completo do SalesInsight PY."""
    print("=" * 60)
    print(" SALESINSIGHT PY - Analise de Dados de Vendas")
    print("=" * 60)

    # Etapa 0 - garantir a existencia do dataset
    if not os.path.exists("vendas.csv"):
        gerar_dataset_vendas("vendas.csv")

    # Etapas 1 a 6 - fluxo pelas funcoes
    registros = carregar_dataset_csv("vendas.csv")
    inspecionar_dados(registros)
    registros_limpos, relatorio = limpar_dados(registros)
    registros_limpos = criar_colunas_derivadas(registros_limpos)
    metricas = calcular_metricas(registros_limpos)
    clientes = segmentar_clientes(registros_limpos)
    exportar_resultados(metricas, clientes)

    print("\n[CONCLUIDO] Fluxo finalizado com sucesso.")


if __name__ == "__main__":
    main()

 SALESINSIGHT PY - Analise de Dados de Vendas
Dataset gerado com 200 registros em vendas.csv.
Processamento de leitura do csv finalizado.

=== INSPECAO INICIAL DO DATASET ===
Total de registros: 200

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Valores ausentes por coluna:
{'id_venda': 0, 'data_venda': 0, 'cliente': 0, 'produto': 0, 'categoria': 0, 'regiao': 0, 'quantidade': 10, 'preco_unitario': 4}

Primeiros registros:
{'id_venda': '1', 'data_venda': '2025-05-21', 'cliente': 'cliente#016', 'produto': 'Mouse', 'categoria': 'Perifericos', 'regiao': 'Sudeste', 'quantidade': '2', 'preco_unitario': '102.9'}
{'id_venda': '2', 'data_venda': '2025-09-16', 'cliente': 'Cliente_039', 'produto': 'Notebook', 'categoria': 'Computadores', 'regiao': 'Norte', 'quantidade': '', 'preco_unitario': '3204.57'}
{'id_venda': '3', 'data_venda': '2025-03-23', 'cliente': 'Cliente_045', 'produto': 'Tablet', 'categoria': 'Celulares', 'regiao': 